# Cell 1: Installations & Imports

In [ ]:
!pip install lightgbm optuna shap fastapi uvicorn python-multipart -q

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
from tqdm.auto import tqdm

import lightgbm as lgb
import optuna
import shap
import joblib
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score,
                             precision_recall_curve, confusion_matrix, log_loss)

optuna.logging.set_verbosity(optuna.logging.WARNING)  

# Cell 2: Config

In [ ]:
DATA_PATH   = r"E:\Courses\depi\depi-mini-and-final-projects\mini-projects\data\processed\brazilian_ecommerce_data.csv"

SEED        = 42
N_TRIALS    = 60          
TIMEOUT_SEC = 20 * 60     
N_SPLITS    = 4          
N_SEEDS     = 5         
TE_SMOOTHING = 20        

# Cell 3: Load Data & Define Target

In [ ]:
df = pd.read_csv(DATA_PATH)

REQUIRED = ['order_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date',
            'order_estimated_delivery_date', 'review_score', 'price', 'freight_value', 'product_category',
            'order_item_id', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty']
missing = [c for c in REQUIRED if c not in df.columns]
assert not missing, f"أعمدة ناقصة في الداتا: {missing}"

date_cols = ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    df[c] = pd.to_datetime(df[c])

df = df[(df['order_status'] == 'delivered') & df['order_delivered_customer_date'].notna() & df['review_score'].notna()].copy()
df['bad_review'] = (df['review_score'] <= 3).astype(int)
df = df.reset_index(drop=True)

START_DATE = df['order_purchase_timestamp'].min()
print(f"Rows: {len(df):,} | Orders: {df['order_id'].nunique():,} | Bad-review rate: {df['bad_review'].mean():.1%}")

# Cell 4: Feature Engineering

In [ ]:
OPTIONAL_NUM = ['payment_installments', 'payment_value', 'product_weight_g', 'product_length_cm',
                'product_height_cm', 'product_width_cm', 'customer_zip_code_prefix', 'seller_zip_code_prefix']
CAT_CANDIDATES = ['product_category', 'customer_state', 'seller_state', 'payment_type']
TE_CANDIDATES  = ['seller_id', 'product_id']          
def create_features(data):
    f = data.copy()
    for c in OPTIONAL_NUM:
        if c in f.columns:
            f[c] = pd.to_numeric(f[c], errors='coerce')

    p = f['order_purchase_timestamp']
    d = f['order_delivered_customer_date']
    e = f['order_estimated_delivery_date']

    f['purchase_month'] = p.dt.month
    f['purchase_day_of_week'] = p.dt.dayofweek
    f['purchase_hour'] = p.dt.hour
    f['is_weekend_purchase'] = f['purchase_day_of_week'].isin([5, 6]).astype(int)
    f['days_since_start'] = (p - START_DATE).dt.total_seconds() / 86400
    f['delivered_day_of_week'] = d.dt.dayofweek

    f['delivery_days'] = (d - p).dt.total_seconds() / 86400
    f['delivery_delay_days'] = (d - e).dt.total_seconds() / 86400
    f['estimated_days'] = (e - p).dt.total_seconds() / 86400
    f['delay_ratio'] = f['delivery_delay_days'] / (f['estimated_days'] + 1)
    f['is_delayed'] = (f['delivery_delay_days'] > 0).astype(int)
    f['early_days'] = (-f['delivery_delay_days']).clip(lower=0)
    f['delivery_to_estimate_ratio'] = f['delivery_days'] / (f['estimated_days'] + 1)
    bins = [-np.inf, 0, 3, 7, 15, np.inf]
    f['delay_severity'] = pd.cut(f['delivery_delay_days'], bins=bins, labels=[0, 1, 2, 3, 4]).astype(int)

    f['price_per_photo'] = f['price'] / (f['product_photos_qty'] + 1)
    f['description_to_name_ratio'] = f['product_description_lenght'] / (f['product_name_lenght'] + 1)
    f['freight_to_price_ratio'] = f['freight_value'] / (f['price'] + 1)
    f['total_value'] = f['price'] + f['freight_value']
    f['product_info_missing'] = f[['product_name_lenght', 'product_description_lenght', 'product_photos_qty']].isna().any(axis=1).astype(int)

    dims = ['product_length_cm', 'product_height_cm', 'product_width_cm']
    if all(c in f.columns for c in dims):
        f['product_volume_cm3'] = f[dims[0]] * f[dims[1]] * f[dims[2]]
    if 'product_weight_g' in f.columns:
        f['freight_per_kg'] = f['freight_value'] / (f['product_weight_g'] / 1000 + 0.1)
        if 'product_volume_cm3' in f.columns:
            f['product_density'] = f['product_weight_g'] / (f['product_volume_cm3'] + 1)

    g = f.groupby('order_id')
    f['order_n_items'] = g['order_id'].transform('size')
    f['order_total_price'] = g['price'].transform('sum')
    f['order_total_freight'] = g['freight_value'].transform('sum')
    f['order_total_value'] = f['order_total_price'] + f['order_total_freight']
    f['price_share'] = f['price'] / (f['order_total_price'] + 1)
    if 'product_id' in f.columns:
        f['order_n_products'] = g['product_id'].transform('nunique')
    if 'seller_id' in f.columns:
        f['order_n_sellers'] = g['seller_id'].transform('nunique')
    if 'customer_state' in f.columns and 'seller_state' in f.columns:
        f['same_state'] = (f['customer_state'] == f['seller_state']).astype(int)
    return f

df_eng = create_features(df)

CAT_COLS = [c for c in CAT_CANDIDATES if c in df_eng.columns]
TE_BASE  = [c for c in TE_CANDIDATES if c in df_eng.columns]
print("Categorical cols       :", CAT_COLS)
print("Target-encoded cols    :", TE_BASE if TE_BASE else "— (مفيش seller_id/product_id في الداتا)")
print("Optional numeric found :", [c for c in OPTIONAL_NUM if c in df_eng.columns] or "—")

# Cell 5: Data Splitting (Preventing Leakage)

In [ ]:
orders = df_eng.groupby('order_id')['bad_review'].max()
tr_ids, tmp_ids = train_test_split(orders.index, test_size=0.3, random_state=SEED, stratify=orders)
val_ids, te_ids = train_test_split(tmp_ids, test_size=0.5, random_state=SEED, stratify=orders.loc[tmp_ids])

tr_mask  = df_eng['order_id'].isin(tr_ids)
val_mask = df_eng['order_id'].isin(val_ids)
te_mask  = df_eng['order_id'].isin(te_ids)
print(f"Train: {tr_mask.sum():,} | Val: {val_mask.sum():,} | Test: {te_mask.sum():,}")

# Cell 6: Leak-free Encodings

In [ ]:
cat_counts = df_eng.loc[tr_mask, 'product_category'].value_counts()
df_eng['category_freq'] = df_eng['product_category'].map(cat_counts).fillna(1)

def add_target_encoding(df, train_mask, cols, smoothing=TE_SMOOTHING, n_splits=5):
    prior = df.loc[train_mask, 'bad_review'].mean()
    tr = df[train_mask]
    maps = {}
    for col in cols:
        st = tr.groupby(col)['bad_review'].agg(['sum', 'count'])
        full = (st['sum'] + prior * smoothing) / (st['count'] + smoothing)
        maps[col] = full.to_dict()

        enc = pd.Series(np.nan, index=df.index)
        enc.loc[~train_mask] = df.loc[~train_mask, col].map(full)
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for a, b in sgkf.split(tr, tr['bad_review'], groups=tr['order_id']):
            st_f = tr.iloc[a].groupby(col)['bad_review'].agg(['sum', 'count'])
            m = (st_f['sum'] + prior * smoothing) / (st_f['count'] + smoothing)
            enc.loc[tr.index[b]] = tr.iloc[b][col].map(m).values
        df[f'te_{col}'] = enc.fillna(prior)     
    return maps, prior

te_maps, te_prior = add_target_encoding(df_eng, tr_mask, TE_BASE)
TE_COLS = [f'te_{c}' for c in TE_BASE]

cat_levels = {}
for c in CAT_COLS:
    s = df_eng[c].astype(object).map(lambda v: v if pd.isna(v) else str(v))
    cat_levels[c] = sorted(s[tr_mask].dropna().unique())
    df_eng[c] = pd.Categorical(s, categories=cat_levels[c])

BASE_NUM = ['price', 'freight_value', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty',
            'order_item_id', 'purchase_month', 'purchase_day_of_week', 'purchase_hour', 'is_weekend_purchase',
            'delivery_days', 'delivery_delay_days', 'estimated_days', 'delay_ratio', 'is_delayed', 'delay_severity',
            'price_per_photo', 'description_to_name_ratio', 'freight_to_price_ratio', 'total_value', 'category_freq']
NEW_NUM  = ['days_since_start', 'delivered_day_of_week', 'early_days', 'delivery_to_estimate_ratio',
            'product_info_missing', 'order_n_items', 'order_total_price', 'order_total_freight', 'order_total_value',
            'price_share', 'order_n_products', 'order_n_sellers', 'same_state',
            'product_volume_cm3', 'freight_per_kg', 'product_density'] + OPTIONAL_NUM

features = [c for c in BASE_NUM + NEW_NUM if c in df_eng.columns] + CAT_COLS + TE_COLS
features = list(dict.fromkeys(features))    

X, y = df_eng[features], df_eng['bad_review']
X_train, y_train = X[tr_mask],  y[tr_mask]
X_val,   y_val   = X[val_mask], y[val_mask]
X_test,  y_test  = X[te_mask],  y[te_mask]
groups_train = df_eng.loc[tr_mask, 'order_id']
print(f"✅ {len(features)} features | categorical: {CAT_COLS} | target-encoded: {TE_COLS}")

# Cell 7: Optuna Tuning (Cross-Validation على الـ Train بس)

In [ ]:
folds = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
             .split(X_train, y_train, groups=groups_train))

def objective(trial):
    param = {
        'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 150),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.5),
        'cat_smooth': trial.suggest_float('cat_smooth', 1, 50),
        'verbose': -1, 'seed': SEED, 'n_jobs': -1,
    }
    scores = []
    for tr_i, va_i in folds:
        dtr = lgb.Dataset(X_train.iloc[tr_i], label=y_train.iloc[tr_i])
        dva = lgb.Dataset(X_train.iloc[va_i], label=y_train.iloc[va_i], reference=dtr)
        m = lgb.train(param, dtr, num_boost_round=2000, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        p = m.predict(X_train.iloc[va_i], num_iteration=m.best_iteration)
        scores.append(average_precision_score(y_train.iloc[va_i], p))     # PR-AUC: الأنسب لـ imbalanced
    return float(np.mean(scores))

print("🚀 Starting Optuna Optimization (metric = mean CV PR-AUC)...")
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.enqueue_trial({'num_leaves': 31, 'max_depth': 8, 'learning_rate': 0.05, 'feature_fraction': 0.8,
                     'bagging_fraction': 0.8, 'bagging_freq': 1, 'min_child_samples': 50,
                     'lambda_l1': 1e-3, 'lambda_l2': 1e-3, 'min_split_gain': 0.0, 'cat_smooth': 10})
study.optimize(objective, n_trials=N_TRIALS, timeout=TIMEOUT_SEC, show_progress_bar=True)

best_params = study.best_params
best_params.update({'objective': 'binary', 'metric': 'auc', 'boosting_type': 'gbdt',
                    'verbose': -1, 'n_jobs': -1})
print(f"\n🏆 Best CV PR-AUC: {study.best_value:.4f}  (after {len(study.trials)} trials)")
print(best_params)

# Cell 8: Train Final Ensemble (Progress Bar + Loss + Accuracy)

In [ ]:
def _loss_acc(preds, data):
    yy = data.get_label()
    p = np.clip(preds, 1e-15, 1 - 1e-15)
    loss = -np.mean(yy * np.log(p) + (1 - yy) * np.log(1 - p))
    acc = np.mean((p >= 0.5) == yy)
    return [('logloss', loss, False), ('accuracy', acc, True)]

class TqdmProgress:
    order = 25
    before_iteration = False
    def __init__(self, total, desc="Training"):
        self.pbar = tqdm(total=total, desc=desc)
    def __call__(self, env):
        tr = {n: v for _, n, v, _ in env.model.eval_train(_loss_acc)}
        va = {n: v for _, n, v, _ in env.model.eval_valid(_loss_acc)}
        self.pbar.set_postfix(ordered_dict=OrderedDict(
            loss=f"{tr['logloss']:.3f}", acc=f"{tr['accuracy']:.3f}",
            val_loss=f"{va['logloss']:.3f}", val_acc=f"{va['accuracy']:.3f}"))
        self.pbar.update(1)
    def close(self):
        self.pbar.total = self.pbar.n
        self.pbar.refresh()
        self.pbar.close()

models = []
for k in range(N_SEEDS):
    params = {**best_params, 'seed': SEED + k, 'bagging_seed': SEED + k, 'feature_fraction_seed': SEED + k}
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval   = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    progress = TqdmProgress(total=3000, desc=f"Model {k + 1}/{N_SEEDS}")
    try:
        m = lgb.train(params, dtrain, num_boost_round=3000, valid_sets=[dtrain, dval],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0), progress])
    finally:
        progress.close()
    models.append(m)
    print(f"   ↳ best iteration = {m.best_iteration}")

def predict_ensemble(Xdf):
    return np.mean([m.predict(Xdf, num_iteration=m.best_iteration) for m in models], axis=0)

# Cell 9: Evaluate

In [ ]:
val_proba  = predict_ensemble(X_val)
test_proba = predict_ensemble(X_test)

precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-8)
best_threshold = float(thresholds[np.argmax(f1_scores[:-1])])
y_pred = (test_proba >= best_threshold).astype(int)

print(f"✅ Best Threshold (from validation): {best_threshold:.3f}")
print(f"📈 Test ROC-AUC: {roc_auc_score(y_test, test_proba):.4f}")
print(f"🎯 Test PR-AUC : {average_precision_score(y_test, test_proba):.4f}")
print(f"📉 Test LogLoss: {log_loss(y_test, test_proba):.4f}   (baseline = {log_loss(y_test, np.full(len(y_test), y_train.mean())):.4f})")
print(f"\nلمقارنة: النتيجة القديمة على داتتك كانت ROC-AUC=0.7022 | PR-AUC=0.5064 | F1(Bad)=0.47")
print(f"(كل موديل لوحده: ROC-AUC = {[round(roc_auc_score(y_test, m.predict(X_test, num_iteration=m.best_iteration)), 4) for m in models]})\n")
print(classification_report(y_test, y_pred, target_names=['Good (4-5)', 'Bad (1-3)']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4.5, 3.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Good', 'Bad'], yticklabels=['Good', 'Bad'])
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix (Test)')
plt.show()

# Cell 10: Explainability

In [ ]:
imp = pd.DataFrame({m_i: m.feature_importance(importance_type='gain') for m_i, m in enumerate(models)}, index=features).mean(axis=1)
imp = (imp / imp.sum()).sort_values(ascending=True).tail(20)
plt.figure(figsize=(9, 7)); imp.plot.barh(color='#4C72B0')
plt.title("Top 20 Features (Gain share, averaged over models)"); plt.tight_layout(); plt.show()

try:
    Xs = X_test.sample(min(3000, len(X_test)), random_state=SEED)
    sv = shap.TreeExplainer(models[0]).shap_values(Xs)
    if isinstance(sv, list):         
        sv = sv[1]
    plt.figure(figsize=(10, 8))
    shap.summary_plot(sv, Xs, max_display=15, plot_type="bar", show=False)
    plt.title("Top 15 Feature Importances (SHAP)", fontsize=14); plt.tight_layout(); plt.show()
except Exception as ex:
    print("⚠️ SHAP skipped:", repr(ex))

# Cell 11: Save Artifacts for API

In [ ]:
bundle = {
    'models': models,                    
    'features': features,
    'threshold': best_threshold,
    'cat_cols': CAT_COLS, 'cat_levels': cat_levels,
    'te_maps': te_maps, 'te_prior': te_prior,
    'cat_freq_map': cat_counts.to_dict(),
    'start_date': START_DATE,
    'best_params': best_params,
}
joblib.dump(bundle, 'bad_review_pipeline.pkl')
for i, m in enumerate(models):
    m.save_model(f'lgbm_bad_review_{i}.txt')
print("💾 Saved: bad_review_pipeline.pkl + lgbm_bad_review_*.txt")